# Cohort Retention Analysis

## Project
**SaaS/E-Commerce Cohort Retention & Customer Lifetime Value (CLTV) Analysis**

## Objective

The objective of this notebook is to analyze customer retention using cohort analysis. Customers are grouped based on the month of their first purchase (cohort month), and their purchasing behavior is tracked over subsequent months to identify retention patterns and customer churn.

## Business Questions

- Which customer cohorts demonstrate the highest retention?
- At which month does customer churn become significant?
- Which acquisition periods generate the most loyal customers?
- What strategic recommendations can improve customer retention?

---

**Author:** Temitope Amos Agboola

**Organization:** Infotact Solutions

**Project Phase:** Week 2 – Cohort Retention Analysis

In [1]:
# ==========================================
# Import Required Libraries
# ==========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Display Options
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# Plot Style
plt.style.use("ggplot")

In [2]:
# ==========================================
# Configure Project Paths
# ==========================================

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"
VISUALS = PROJECT_ROOT / "visuals"

In [3]:
# ==========================================
# Load Clean Dataset
# ==========================================

df = pd.read_csv(
    PROCESSED_DATA / "online_retail_clean.csv",
    parse_dates=["InvoiceDate"]
)

print(f"Dataset Shape: {df.shape}")

df.head()

Dataset Shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


In [4]:
# ==========================================
# Dataset Information
# ==========================================

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  str           
 1   StockCode    1067371 non-null  str           
 2   Description  1062989 non-null  str           
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 65.1 MB


In [5]:
df.describe(include="all")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
count,1067371,1067371,1062989,"1,067,371.00",1067371,"1,067,371.00","824,364.00",1067371
unique,53628,5305,5698,NaN,NaN,NaN,NaN,43
top,537434,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,United Kingdom
freq,1350,5829,5918,NaN,NaN,NaN,NaN,981330
mean,NaN,NaN,NaN,9.94,2011-01-02 21:13:55.394029,4.65,"15,324.64",NaN
min,NaN,NaN,NaN,"-80,995.00",2009-12-01 07:45:00,"-53,594.36","12,346.00",NaN
25%,NaN,NaN,NaN,1.00,2010-07-09 09:46:00,1.25,"13,975.00",NaN
50%,NaN,NaN,NaN,3.00,2010-12-07 15:28:00,2.10,"15,255.00",NaN
75%,NaN,NaN,NaN,10.00,2011-07-22 10:23:00,4.15,"16,797.00",NaN
max,NaN,NaN,NaN,"80,995.00",2011-12-09 12:50:00,"38,970.00","18,287.00",NaN


In [6]:
# ==========================================
# Create Invoice Month
# ==========================================

df["InvoiceMonth"] = df["InvoiceDate"].dt.to_period("M")

df[["InvoiceDate", "InvoiceMonth"]].head()

,InvoiceDate,InvoiceMonth
0,2009-12-01 07:45:00,2009-12
1,2009-12-01 07:45:00,2009-12
2,2009-12-01 07:45:00,2009-12
3,2009-12-01 07:45:00,2009-12
4,2009-12-01 07:45:00,2009-12


## Create Customer Cohort Month

Each customer is assigned to the month of their first purchase. This month is known as the **Cohort Month** and represents the customer's acquisition period.

The Cohort Month remains constant for all future purchases made by the same customer.

In [8]:
# ==========================================
# Create Cohort Month (First Purchase Month)
# ==========================================

# Determine each customer's first purchase month
cohort = (
    df.groupby("Customer ID")["InvoiceMonth"]
      .min()
      .rename("CohortMonth")
)

# Merge Cohort Month back into the dataset
df = df.merge(cohort, on="Customer ID")

# Preview
df[["Customer ID", "InvoiceMonth", "CohortMonth"]].head(10)

,Customer ID,InvoiceMonth,CohortMonth
0,"13,085.00",2009-12,2009-12
1,"13,085.00",2009-12,2009-12
2,"13,085.00",2009-12,2009-12
3,"13,085.00",2009-12,2009-12
4,"13,085.00",2009-12,2009-12
5,"13,085.00",2009-12,2009-12
6,"13,085.00",2009-12,2009-12
7,"13,085.00",2009-12,2009-12
8,"13,085.00",2009-12,2009-12
9,"13,085.00",2009-12,2009-12


## Create Cohort Index

The Cohort Index measures the number of months since a customer's first purchase.

- Month 1 = Acquisition Month
- Month 2 = One month after acquisition
- Month 3 = Two months after acquisition

This metric allows customers from different acquisition periods to be compared on the same timeline.

In [9]:
# ==========================================
# Create Cohort Index
# ==========================================

invoice_year = df["InvoiceMonth"].dt.year
invoice_month = df["InvoiceMonth"].dt.month

cohort_year = df["CohortMonth"].dt.year
cohort_month = df["CohortMonth"].dt.month

df["CohortIndex"] = (
    (invoice_year - cohort_year) * 12
    + (invoice_month - cohort_month)
    + 1
)

df[[
    "Customer ID",
    "InvoiceMonth",
    "CohortMonth",
    "CohortIndex"
]].head(10)

,Customer ID,InvoiceMonth,CohortMonth,CohortIndex
0,"13,085.00",2009-12,2009-12,1
1,"13,085.00",2009-12,2009-12,1
2,"13,085.00",2009-12,2009-12,1
3,"13,085.00",2009-12,2009-12,1
4,"13,085.00",2009-12,2009-12,1
5,"13,085.00",2009-12,2009-12,1
6,"13,085.00",2009-12,2009-12,1
7,"13,085.00",2009-12,2009-12,1
8,"13,085.00",2009-12,2009-12,1
9,"13,085.00",2009-12,2009-12,1


In [10]:
# ==========================================
# Validate Cohort Features
# ==========================================

print("Unique Cohorts:", df["CohortMonth"].nunique())
print("Minimum Cohort Index:", df["CohortIndex"].min())
print("Maximum Cohort Index:", df["CohortIndex"].max())

df[["InvoiceMonth", "CohortMonth", "CohortIndex"]].describe(include="all")

Unique Cohorts: 25
Minimum Cohort Index: 1
Maximum Cohort Index: 25


,InvoiceMonth,CohortMonth,CohortIndex
count,824364,824364,"824,364.00"
unique,25,25,NaN
top,2011-11,2009-12,NaN
freq,65598,345402,NaN
mean,NaN,NaN,9.15
std,NaN,NaN,7.19
min,NaN,NaN,1.00
25%,NaN,NaN,2.00
50%,NaN,NaN,8.00
75%,NaN,NaN,14.00
